# Chapter 3 — Self-Attention Q&A Notes

Working notes from questions asked while going through `listing_3_1_class.py` / `listing_3_2_class.py` (simplified self-attention, Chapter 3 of *Build a Large Language Model From Scratch*).

Run this notebook from the `chapter_3/` folder so the local imports resolve.

In [ ]:
import torch
from listing_3_1_class import SelfAttention_v1
from listing_3_2_class import SelfAttention_v2

torch.manual_seed(123)

d_in = 3
d_out = 2

inputs = torch.tensor([[0.43, 0.15, 0.89],  # Your
                       [0.55, 0.87, 0.66],  # Journey
                       [0.57, 0.85, 0.64],  # starts
                       [0.22, 0.58, 0.33],  # with
                       [0.77, 0.25, 0.10],  # one
                       [0.05, 0.80, 0.55]]) # step

inputs.shape

## Q: What is the interpretation of the context vector?

`context_vec = attn_weights @ values` — each row is the context vector for one token.

It's the original token re-expressed as a **weighted blend of every token's `value` vector in the sequence** (including itself). The weights are the softmax attention weights: the more relevant another token is, the more its `value` contributes.

- **Input embedding** = the word's static, context-free meaning.
- **Context vector** = the word's meaning enriched by its surrounding sentence, weighted by attention.

This is exactly what lets the model disambiguate the same word differently depending on context.

In [ ]:
sa_v1 = SelfAttention_v1(d_in, d_out)
print("Context vectors:\n", sa_v1(inputs))

## Q: What is the dimension of the weight matrix?

`W_query`, `W_key`, `W_value` are each `(d_in, d_out)` = `(3, 2)` here.

For `x @ W` to work, where `x` is `(num_tokens, d_in)`, the matrix-multiply rule requires `W`'s first dimension to match `d_in`; the result's width becomes `W`'s last dimension (`d_out`):

```
(6, 3) @ (3, 2) = (6, 2)
```

Rows of `W` = `d_in` (input embedding size). Columns of `W` = `d_out` (projected size). The three matrices share this shape but are independently learned.

In [ ]:
print("W_query shape:", sa_v1.W_query.shape)
print(sa_v1.W_query)
print("\nW_key shape:", sa_v1.W_key.shape)
print(sa_v1.W_key)
print("\nW_value shape:", sa_v1.W_value.shape)
print(sa_v1.W_value)

## Q: What's the shape of a single token's query (e.g. `x_2` = "Journey")?

Depends entirely on *how you slice* the token — a 1D PyTorch tensor has **no row/column orientation** at all, unlike a classic linear-algebra column vector.

- `inputs[1]` → shape `(3,)`, a plain 1D tensor. `x_2 @ W_query` → shape `(2,)`, also 1D.
- `inputs[1:2]` → shape `(1, 3)`, explicitly 2D. `x_2 @ W_query` → shape `(1, 2)`.

Neither is "wrong"; `torch.matmul` has a special rule for 1D-by-2D multiplication that treats the 1D tensor as a row for the multiply, but returns a 1D result — not literally `2x1`.

In [ ]:
x_2 = inputs[1]
print("x_2 (1D):", x_2, x_2.shape)
q_2 = x_2 @ sa_v1.W_query
print("query_2 (1D):", q_2, q_2.shape)

x_2_row = inputs[1:2]
print("\nx_2_row (2D):", x_2_row, x_2_row.shape)
q_2_row = x_2_row @ sa_v1.W_query
print("query_2_row (2D):", q_2_row, q_2_row.shape)

## Q: How exactly is `query_2 = x_2 @ W_query` computed?

It's two dot products — one per column of `W_query`. Each output element sums 3 products (one per input dimension, `d_in=3`):

```
query_2[0] = x_2[0]*W_query[0,0] + x_2[1]*W_query[1,0] + x_2[2]*W_query[2,0]
query_2[1] = x_2[0]*W_query[0,1] + x_2[1]*W_query[1,1] + x_2[2]*W_query[2,1]
```

In [ ]:
manual_col0 = x_2[0]*sa_v1.W_query[0,0] + x_2[1]*sa_v1.W_query[1,0] + x_2[2]*sa_v1.W_query[2,0]
manual_col1 = x_2[0]*sa_v1.W_query[0,1] + x_2[1]*sa_v1.W_query[1,1] + x_2[2]*sa_v1.W_query[2,1]

print("manual query_2:", manual_col0.item(), manual_col1.item())
print("x_2 @ W_query: ", q_2)

## Q: During self-attention, do only the inputs change, while the weight matrices stay the same for the whole sentence?

Yes. Within one forward pass, `W_query`/`W_key`/`W_value` are **fixed and shared across every token**. Only `x_i` changes per token.

```
inputs (6, 3)  @  W_query (3, 2)  =  queries (6, 2)
  ↑ 6 different       ↑ same matrix      ↑ 6 different
    token vectors        for every row      query vectors
```

This weight sharing is what makes attention position-independent. The matrices only change across *training steps* (via backprop), not while processing a single sentence.

## Q: At the start, are all the weight matrices random?

Yes — `W_query`, `W_key`, `W_value` (and every other trainable matrix: embeddings, feed-forward, output layer) start as random numbers with zero learned information.

- Not "pure" naive random: PyTorch uses specific init schemes (`nn.Linear` defaults to Kaiming/uniform scaled by layer size; `torch.rand` used in `SelfAttention_v1` is plain uniform `[0,1)`).
- `torch.manual_seed(123)` doesn't remove randomness, it just makes it *reproducible* across runs.
- Meaningful structure (similar words → similar vectors, attention learning what to focus on) only emerges through training.

## Q: When calling `sa_v2.W_query(inputs)` (an `nn.Linear`), are we skipping the dot product? Is `nn.Linear` an improvement over it?

No — it's the *same* dot products, just packaged differently. `nn.Linear.__call__` internally computes `x @ weight.T (+ bias)`.

In [ ]:
import torch.nn as nn

torch.manual_seed(123)
lin = nn.Linear(3, 2, bias=False)
x = torch.tensor([0.55, 0.87, 0.66])

print("lin.weight shape (out_features, in_features):", lin.weight.shape)

out_via_call = lin(x)
out_via_matmul = x @ lin.weight.T

print("lin(x):       ", out_via_call)
print("x @ weight.T: ", out_via_matmul)
print("identical?", torch.allclose(out_via_call, out_via_matmul))

The real differences between `SelfAttention_v1` (`nn.Parameter` + manual `@`) and `SelfAttention_v2` (`nn.Linear`) are not mathematical:

1. **Weight storage convention** — `nn.Linear`'s weight is `(out_features, in_features)`, the transpose of v1's `(d_in, d_out)`. That's why `forward()` calls `self.W_key(x)` instead of `x @ self.W_key`.
2. **Better default initialization** — Kaiming-uniform (scaled by `1/√fan_in`) vs. plain `torch.rand(0,1)`.
3. **Optional bias term** — `qkv_bias=True` adds a learnable bias, which plain `nn.Parameter` in v1 doesn't have built in.
4. **Framework integration** — auto-registers as a submodule, shows up in `model.parameters()`, works with `.to(device)`, etc.